# TP 9 — Soumettre, mesurer, expliquer

[![Ouvrir dans Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/earthai-tech/lab-courses/blob/main/curat/data-science/notebooks/TP9_Soumettre_mesurer_expliquer.ipynb)

**Data Sciences, visualisation et Big Data** · M2 Télédétection et SIG ·
Séance 9 · KOUADIO Kouao Laurent

Trois parties :
- **A. Sur le supercalculateur** (si vous avez un compte) : lire
  `sinfo`, soumettre un job SLURM, le suivre.
- **B. Sur votre portable** : une petite mesure d'accélération avec
  Dask.
- **C. Analyse** : accélération, efficacité et loi d'Amdahl à partir
  des mesures réelles du supercalculateur.

Durée : environ 90 minutes.

In [ ]:
# Google Colab : installe les paquets du cours (environ 1 minute).
# Dans l'environnement conda du cours (environment.yml), rien à faire.
import sys
if "google.colab" in sys.modules:
    %pip install -q pystac-client odc-stac

## A. Sur le supercalculateur (terminal)

Ces commandes se tapent dans un terminal, pas dans le notebook.

```bash
ssh hpc                         # connexion au nœud de connexion
sinfo -s                        # état des partitions
cd ~/projects/geodatasciences
cat s9_local.sbatch             # lire le script avant de le lancer
sbatch s9_local.sbatch          # soumettre ; note le numéro du job
squeue -u $USER                 # PD = en attente, R = en cours
sacct -j <numéro> --format=JobID,State,Elapsed,MaxRSS
```

**Question 1.** Dans `sinfo -s`, que signifient les quatre nombres de
`NODES(A/I/O/T)` ? Combien de nœuds B720 sont libres maintenant ?

**Question 2.** Pourquoi les nœuds de calcul ne peuvent-ils pas
télécharger les images eux-mêmes ? Comment a-t-on contourné le
problème (voir `s9_stage.py`) ?

## B. Sur votre portable : Dask avec 1, 2 et 4 workers

On reprend le cube NDVI de la séance 4 (6 scènes, 60 m), lu
directement dans le cloud.

In [ ]:
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xarray as xr
from dask.distributed import Client, LocalCluster
from odc.stac import configure_rio, load
from pystac_client import Client as StacClient

configure_rio(cloud_defaults=True, GDAL_HTTP_MAX_RETRY="10")
cat = StacClient.open("https://earth-search.aws.element84.com/v1")
items = cat.search(collections=["sentinel-2-l2a"],
                   bbox=[-4.10, 5.25, -3.85, 5.45],
                   datetime="2025-01-01/2025-03-31",
                   query={"eo:cloud_cover": {"lt": 60}},
                   max_items=6).item_collection()
print(len(items), "scènes")


def mediane_ndvi():
    ds = load(items, bands=["red", "nir", "scl"],
              bbox=[-4.10, 5.25, -3.85, 5.45], crs="EPSG:32630",
              resolution=60, chunks={"x": 256, "y": 256})
    ok = ds.scl.isin([4, 5, 6, 11])
    nd = ((ds.nir - ds.red) / (ds.nir + ds.red)).where(ok)
    return nd.median("time")


temps = {}
for n in (1, 2, 4):
    with LocalCluster(n_workers=n, threads_per_worker=1,
                      dashboard_address=None) as cl, Client(cl):
        t0 = time.time()
        mediane_ndvi().compute()
        temps[n] = time.time() - t0
        print(f"{n} worker(s) : {temps[n]:.1f} s")

**Question 3.** L'accélération est-elle proche de ×4 avec 4 workers ?
Sur votre portable, qu'est-ce qui limite : les cœurs ou le réseau ?

## C. Analyse des mesures du supercalculateur

Médiane NDVI de 24 dates sur toute la tuile 30NUL (20 m), données
déjà copiées sur Lustre. Mesures réelles, novembre 2026 : 1 à 28
cœurs sur un nœud (`local-…`), puis 2 et 4 nœuds de 28 cœurs
(`slurm-…`, avec dask-jobqueue).

In [ ]:
mesures = pd.DataFrame([{"mode": "local-1", "coeurs": 1, "secondes": 420.7}, {"mode": "local-4", "coeurs": 4, "secondes": 113.5}, {"mode": "local-7", "coeurs": 7, "secondes": 71.6}, {"mode": "local-14", "coeurs": 14, "secondes": 56.4}, {"mode": "local-28", "coeurs": 28, "secondes": 59.3}, {"mode": "slurm-2-noeuds", "coeurs": 56, "secondes": 42.9}, {"mode": "slurm-4-noeuds", "coeurs": 112, "secondes": 34.9}])
t1 = mesures.loc[mesures.coeurs == 1, "secondes"].iloc[0]
mesures["acceleration"] = t1 / mesures["secondes"]
mesures["efficacite"] = mesures["acceleration"] / mesures["coeurs"]
mesures.round(2)

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(11, 4.2))
p = mesures["coeurs"]
axs[0].plot(p, mesures["acceleration"], "o-", color="#E07A1F",
            label="mesurée")
axs[0].plot(p, p, "--", color="grey", label="idéale")
axs[0].set_xlabel("cœurs")
axs[0].set_ylabel("accélération")
axs[0].legend()
axs[1].plot(p, 100 * mesures["efficacite"], "o-", color="#1B4332")
axs[1].set_xlabel("cœurs")
axs[1].set_ylabel("efficacité (%)")
axs[1].set_ylim(0, 105)
for ax in axs:
    ax.set_xscale("log", base=2)
    ax.spines[["top", "right"]].set_visible(False)
plt.show()

### Loi d'Amdahl

$$S(p) = \frac{1}{(1 - f) + f / p}$$

On cherche la part parallélisable $f$ qui explique le mieux les
mesures (moindres carrés).

In [ ]:
from scipy.optimize import curve_fit


def amdahl(p, f):
    return 1 / ((1 - f) + f / p)


(f,), _ = curve_fit(amdahl, mesures["coeurs"],
                    mesures["acceleration"], p0=[0.9],
                    bounds=(0, 1))
print(f"f = {f:.3f} ; accélération maximale 1/(1-f) = "
      f"{1 / (1 - f):.0f}")

pp = np.logspace(0, np.log2(512), 50, base=2)
plt.plot(mesures["coeurs"], mesures["acceleration"], "o",
         color="#E07A1F", label="mesures")
plt.plot(pp, amdahl(pp, f), color="#1B4332", label=f"Amdahl, f={f:.3f}")
plt.xscale("log", base=2)
plt.xlabel("cœurs")
plt.ylabel("accélération")
plt.legend()
plt.show()

**Question 4.** Avec ce $f$, combien de cœurs faut-il pour obtenir
×50 ? Est-ce possible ? Que faudrait-il changer dans le calcul ?

**Question 5.** Rédigez une conclusion de 5 lignes : vaut-il mieux
traiter cette tuile sur un portable, sur 1 nœud, ou sur 4 nœuds ?
Tenez compte du temps d'attente dans la file SLURM et du temps de
copie des données.

## À rendre

Le notebook exécuté (parties B et C) et vos réponses.